In [3]:
def fibonacci_retracement_levels(high, low):
    """
    计算斐波那契回撤水平
    :param high: 价格的高点
    :param low: 价格的低点
    :return: 斐波那契回撤水平的列表
    """
    # levels = [0.236, 0.382, 0.5, 0.618, 0.786, 1.0, 1.618, 2.618, 4.236, 6.854, 11.069, 17.803]
    levels = [0.236, 0.382, 0.5, 0.618, 0.786]
    retracement_levels = []
    
    for level in levels:
        retracement_level = high - (high - low) * level
        retracement_levels.append(retracement_level)
    
    return retracement_levels

###### 斐波那契回撤，是技术分析中常用的一种工具，用于识别潜在的支撑和阻力水平。它基于斐波那契数列中的比例关系，主要包括以下几个关键水平：
- 23.6% 
- 38.2%
- 50%
- 61.8%
- 78.6%
- 100%
- 161.8%
- 261.8%
- 423.6%
- 685.4%
- 1106.9%
- 1780.3%
- 这些水平是通过将价格的高点和低点之间的差距乘以相应的斐波那契比例来计算的。交易者通常使用这些水平来预测价格可能会遇到的支撑或阻力区域，从而做出买入或卖出的决策。例如，如果价格从一个高点回撤到一个低点，交易者可能会观察这些斐波那契回撤水平，以确定价格可能会反弹的区域。相反，如果价格从一个低点反弹到一个高点，交易者可能会观察这些水平，以确定价格可能会遇到阻力的区域。总之，斐波那契回撤是一个重要的工具，可以帮助交易者识别潜在的市场转折点，并制定相应的交易策略。

###### 斐波那契回撤水平的计算方法如下：1. **确定高点和低点**：首先，交易者需要确定价格的高点和低点。这可以是一个特定时间段内的最高价和最低价，或者是一个特定趋势中的最高价和最低价。2. **计算价格差距**：计算高点和低点之间的价格差距，即：价格差距 = 高点 - 低点。3. **应用斐波那契比例**：将价格差距乘以斐波那契比例，以计算各个回撤水平的价格。例如，计算23.6%回撤水平的价格可以使用以下公式：23.6%回撤水平价格 = 高点 - (价格差距 * 0.236)。同样，计算其他回撤水平的价格也可以使用类似的公式，只需将0.236替换为相应的斐波那契比例即可。4. **绘制回撤水平**：将计算出的回撤水平价格绘制在图表上，以便交易者可以直观地看到这些潜在的支撑和阻力水平。通过观察价格在这些回撤水平附近的行为，交易者可以做出更明智的交易决策。例如，如果价格在某个回撤水平附近出现反转迹象，交易者可能会考虑进入或退出市场。总之，斐波那契回撤水平的计算方法是基于价格的高点和低点之间的差距，并应用斐波那契比例来确定潜在的支撑和阻力水平。这些水平可以帮助交易者识别市场转折点，并制定相应的交易策略。


In [4]:
def risk_reward_ratio(entry_price, stop_loss_price, take_profit_price):
    """
    计算风险回报比
    :param entry_price: 进场价格
    :param stop_loss_price: 止损价格
    :param take_profit_price: 止盈价格
    :return: 风险回报比
    """
    risk = entry_price - stop_loss_price
    reward = take_profit_price - entry_price
    
    if risk == 0:
        return float('inf')  # 避免除以零的情况
    
    return {"Risk-Reward Ratio": reward / risk, "Risk": risk, "Reward": reward}

###### 从函数里面拿到风险比，风险，收益。
###### 输入可承受风险，根据斐波那契回撤不同水平挂单买入不同数量，根据斐波那契回撤不同水平挂单卖出不同数量。
###### 统一止损价格，盈利目标可以不一样。


##### 通过凸优化，解出每个价格的开单数量。


In [8]:
import cvxpy as cp
import numpy as np

# 先根据 swing high / low 生成斐波那契回撤价位。
high = 2464
low = 2174
entry_price = np.array(fibonacci_retracement_levels(high, low), dtype=float)


def optimize_position_size(entry_price, stop_loss_price, take_profit_price, max_risk):
    """
    通过凸优化计算每个价格的开单数量。

    entry_price = 每个挂单价位的进场价格数组。
    stop_loss_price = 统一止损价。
    take_profit_price = 每个价位对应的止盈价；也可以传单个标量。
    max_risk = 总风险上限，定义为全部成交后打到止损的最大亏损金额。

    返回值 = 每个价位的最优开单数量。
    """
    entry_price = np.asarray(entry_price, dtype=float)
    if len(entry_price) != 5:
        raise ValueError("当前版本固定要求 5 个开单点")

    # 如果止盈价传入的是单个数字，就扩展成和 entry_price 等长的数组。
    if np.isscalar(take_profit_price):
        take_profit_price = np.full_like(entry_price, float(take_profit_price))
    else:
        take_profit_price = np.asarray(take_profit_price, dtype=float)

    # risk = 每买 1 单位、打到止损时亏多少钱。
    # reward = 每买 1 单位、打到止盈时赚多少钱。
    risk = entry_price - float(stop_loss_price)
    reward = take_profit_price - entry_price

    # 这里不再过滤档位，而是要求 5 个档位都合法。
    # 因为每一档都有固定的数量上下界，过滤后会把约束位置打乱。
    if np.any(risk <= 0):
        raise ValueError("存在 entry_price <= stop_loss_price 的档位，当前模型无法求解")
    if np.any(reward <= 0):
        raise ValueError("存在 take_profit_price <= entry_price 的档位，当前模型无法求解")

    # 5 个开单点的数量限制。
    lower_bounds = np.array([0.5, 0.75, 0.5, 0.5, 0.0], dtype=float)
    upper_bounds = np.array([1.0, 1.25, 1.5, 1.5, np.inf], dtype=float)

    # x = 决策变量向量；x[i] 表示第 i 个价位的开单数量。
    x = cp.Variable(len(entry_price))

    # 约束含义：
    # x[0] in [0.5, 1.0]
    # x[1] in [0.75, 1.25]
    # x[2] in [0.5, 1.5]
    # x[3] in [0.5, 1.5]
    # x[4] >= 0，且不设上限
    # 同时满足总风险不超过 max_risk。
    constraints = [
        x >= lower_bounds,
        x[:4] <= upper_bounds[:4],
        risk @ x <= float(max_risk),
    ]

    # 目标函数：最大化总收益 sum(reward[i] * x[i])。
    # 这里不能写 reward / risk，因为那会把变量约掉，失去优化意义。
    objective = cp.Maximize(reward @ x)

    problem = cp.Problem(objective, constraints)
    problem.solve()

    if problem.status not in ("optimal", "optimal_inaccurate"):
        raise ValueError(f"优化失败，problem.status = {problem.status}")

    return x.value


# 示例：每个价位给一个对应止盈价，然后求最优数量。
take_profit_price = np.array([2440, 2420, 2400, 2380, 2360], dtype=float)
max_risk = 500
stop_loss_price = 2170

print(optimize_position_size(entry_price, stop_loss_price, take_profit_price, max_risk))
print(entry_price)

[0.5        0.75       0.5        0.5        1.78496821]
[2395.56 2353.22 2319.   2284.78 2236.06]


##### 这一版，因为每个价位没有做区分，求凸优化会天然的趋于最后一个价位。
##### 改进点：1，给每个价位，拿到到达改价位的概率，这个概率是联合分布的，是一个区间的概率。所以，开单数量应该是一个区间的数量。2，同一个模型，不仅会输出比当前价格小的价格区间的概率，还会输出比当前价格大的价格区间的概率。这个就是盈利目标的概率。3，风险和收益的计算应该是区间的风险和区间的收益。4，某区间的盈利目标应该拿模型来预测当前价格达到盈利目标的概率。
